# Results for model: microsoft/phi-3-mini-128k-instruct

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import cudf

# Define a function to calculate rolling quantiles
def calculate_rolling_quantile(df, feature, target, window, quantile):
    df['rolling_mean'] = df.groupby(['date_id'])['feature'].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    df['rolling_sum'] = df.groupby(['date_id'])['feature'].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
    df['rolling_count'] = df.groupby(['date_id'])['feature'].rolling(window, min_periods=1).count().reset_index(level=0, drop=True)
    
    df['rolling_feature'] = df['rolling_mean'] + quantile * (df['rolling_sum'] - df['rolling_mean']) / df['rolling_count']
    return df['rolling_feature']

# Define a function to load data and prepare the dataset
def get_features_and_target(filepath):
    train_df = pl.read_parquet(filepath)
    group_by = ['date_id', 'household_key']
    
    # Calculate rolling quantile for 'feature_00' based on 'responder_6'
    top_quantile_threshold = train_df.groupby('responder_6').agg(pl.col('feature_00').quantile(0.85)).to_numpy()
    
    # Create mask for top 15% (assuming top 1 elementary quantile as the high end for top 15%)
    mask = train_df['feature_00'] >= top_quantile_threshold[train_df['responder_6']]
    
    train_df['target'] = train_df['responder_6'].astype(np.float32)
    features = train_df[features_cols].to_numpy()
    target = train_df['target'].to_numpy().flatten()
    
    features, target = features[mask], target[mask]
    
    # Create rolling batches
    window_size = 7  # This can be a hyperparameter or treated as a hyperparameter search space
    batch_targets, batch_features = [], []
    
    for i in range(len(target) - window_size + 1):
        batch_targets.append(target[i:i+window_size])
        batch_features.append(features[i:i+window_size])
        
    return np.array(batch_features), np.array(batch_targets)

# Load features and target
features_cols = ['feature_00', 'responder_6']  # Add all other necessary feature columns
data_path = './jane_street_data/train.parquet'
X, y = get_features_and_target(data_path)

# Split the data into training and validation sets
split_index = int(len(X) * 0.8)
X_train, X_valid = X[:split_index], X[split_index:]
y_train, y_valid = y[:split_index], y[split_index:]

# Define the XGBoost regressor
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'predictor': 'gpu_predictor'
}

# Train XGBoost model
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
watchlist = [(dtrain, 'train'), (dvalid, 'eval')]
model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, early_stopping_rounds=10)

print("Training complete")